In [1]:
import requests

api_url = "https://115.159.159.82/api/devices"

r = requests.get(
    api_url,
    params={
        "page": 1,
        "page_size": 50,
    },
    timeout=10
)

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json


In [2]:
data = r.json()

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['page', 'page_size', 'total', 'items'])


In [3]:
print(data)

{'page': 1, 'page_size': 50, 'total': 5000, 'items': [{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}, {'device_id': 'D00000002', 'company_id': 'C000001', 'device_name': '循环泵-000001-002', 'device_type': 'pump', 'model': 'SP-PUM-500', 'zone': '智造一区', 'install_date': '2016-03-06', 'rated_power_kw': 73.8, 'sampling_interval_s': 60, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-08T05:44:00'}, {'device_id': 'D00000003', 'company_id': 'C000001', 'device_name': '空压机-000001-003', 'device_type': 'compressor', 'model': 'SP-COM-200', 'zone': '智造一区', 'install_date': '2016-12-18', 'rated_power_kw': 248.6, 'sampling_interval_s': 30, 'protocol': 'modbus', 'status': 'running', 'updated_at': '2026-08-29T04:01:00'}, {'device_id': 'D

In [4]:
total = data["total"]
page_size = 50
total_pages = (total + page_size - 1) // page_size


In [5]:
total = data["total"]
page_size = data["page_size"]

total_pages = (total + page_size - 1) // page_size

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)


total: 5000
page_size: 50
total_pages: 100


In [6]:
devices = []

for page in range(1, total_pages + 1):
    r = requests.get(
        api_url,
        params={
            "page": page,          # ← 循环变量在这
            "page_size": page_size,
        },
        timeout=10
    )
    page_data = r.json()

    # TODO: 把本页设备加入 devices
    devices.extend(page_data["items"]) 
print("device count:", len(devices))


device count: 5000


In [8]:
from bs4 import BeautifulSoup

base_url = "https://115.159.159.82"
current_url = base_url + "/companies"

r = requests.get(current_url, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

cards = soup.select("article.company-card")
print("本页企业数：", len(cards))


本页企业数： 20


In [9]:
cards = soup.select("article.company-card")

card = cards[0]

company_id = card.get("data-company-id")
name = card.select_one("h2 a").get_text(strip=True)
status = card.select_one("span.badge").get_text(strip=True)


In [10]:
details = {}
for item in card.select("dl > div"):
    details[item.select_one("dt").get_text(strip=True)] = item.select_one("dd").get_text(strip=True)

industry     = details["所属行业"]
zone         = details["所在区域"]
device_count = int(details["设备数量"].replace("台", "").strip())

print(company_id, "|", name, "|", status, "|", industry, "|", zone, "|", device_count)


C000001 | 临港精工001有限公司 | 运营中 | 汽车零部件 | 智造一区 | 6


In [12]:
next_link = soup.select_one("nav.pagination a:last-child")

print(next_link.get_text(strip=True))
print(next_link.get("href"))



下一页
/companies?page=2


In [14]:
def parse_card(card):
    details = {}
    for item in card.select("dl > div"):
        details[item.select_one("dt").get_text(strip=True)] = item.select_one("dd").get_text(strip=True)

    return {
        "company_id":   card.get("data-company-id"),
        "name":         card.select_one("h2 a").get_text(strip=True),
        "status":       card.select_one("span.badge").get_text(strip=True),
        "industry":     details["所属行业"],
        "zone":         details["所在区域"],
        "device_count": int(details["设备数量"].replace("台", "").strip()),
    }


companies = []
current_url = base_url + "/companies"

for i in range(3):
    r = requests.get(current_url, timeout=10)
    page_soup = BeautifulSoup(r.text, "html.parser")

    page_cards = page_soup.select("article.company-card")
    companies.extend(parse_card(c) for c in page_cards)

    next_link = page_soup.select_one("nav.pagination a:last-child")
    if next_link is None:
        break
    current_url = urljoin(current_url, next_link.get("href"))

print("companies:", len(companies))
print("第一条：", companies[0])



companies: 60
第一条： {'company_id': 'C000001', 'name': '临港精工001有限公司', 'status': '运营中', 'industry': '汽车零部件', 'zone': '智造一区', 'device_count': 6}


In [15]:
# 基本验证
print("companies:", len(companies))
print("unique company_id:", len(set(x["company_id"] for x in companies)))

print("devices:", len(devices))
print("unique device_id:", len(set(x["device_id"] for x in devices)))

# 一致性比对：每家企业的 device_count vs Device API 里实际属于它的设备数
mismatch_count = 0
for x in companies:
    api_count = sum(1 for d in devices if d["company_id"] == x["company_id"])
    if x["device_count"] != api_count:
        mismatch_count += 1
        print("不一致:", x["company_id"], "页面 =", x["device_count"], "API =", api_count)

print("不一致企业数:", mismatch_count)


companies: 60
unique company_id: 60
devices: 5000
unique device_id: 5000
不一致企业数: 0


In [17]:
from pathlib import Path
import json

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")


数据已保存到 data/ 目录


In [18]:
import os
from pathlib import Path

print("当前目录:", os.getcwd())
print("data 是否存在:", Path("data").exists())
print("data 绝对路径:", Path("data").resolve())
print("data 里的文件:", os.listdir("data") if Path("data").exists() else "不存在")


当前目录: /home/pwuser/workspace
data 是否存在: True
data 绝对路径: /home/pwuser/workspace/data
data 里的文件: ['companies.json', 'devices.json']


In [19]:
data_dir = Path("L01_source_discovery/data")
data_dir.mkdir(parents=True, exist_ok=True)
